# BPI Challenge 2013: incidents — extended evaluation

This notebook examines how investigation priorities change with stakeholder weights, small-group adjustment, scoring mode and whole-case context. It also checks evidence coverage and explains the signed contribution of each layer. The installed library computes the results; the notebook selects and plots them.

The original norm and reported values provide the baseline. The additional figures show this notebook’s sensitivity analyses; [WISE_Evaluation.ipynb](WISE_Evaluation.ipynb) contains Figures 4–6.


## PM² analysis route and journal

The analysis follows [PM²](https://vdaalst.com/publications/p824.pdf), with iterations between Data Processing, Mining & Analysis and Evaluation.

| Stage | Work in this notebook | Completion boundary |
|---|---|---|
| 1. Planning | Specify investigation questions, case unit and declared scenarios | Process-owner goals and norm approval remain pending |
| 2. Extraction | Verify the supplied public benchmark input | No new organisational extraction is claimed |
| 3. Data Processing | Call the source adapter and retain complete cases and context | Original trace retrieval and time cohorts remain extensions |
| 4. Mining & Analysis | Case assessment, prioritisation, scope/evidence diagnostics and sensitivity | This is process analytics; no newly discovered control-flow model is claimed |
| 5. Evaluation | Compare printed values, test arithmetic and invariants, diagnose concerns | Source/system witness verification and stakeholder validation remain pending |
| 6. Process Improvement & Support | Form evidence-led actions and an implementation backlog | No intervention, realised saving or live support is claimed |

Before treating an assessed concern as a process defect, confirm the norm with the process owner and inspect the source evidence.


In [ ]:
import os
import json
import hashlib
import math
from pathlib import Path
from itertools import combinations
from dataclasses import replace
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

try:
    import wise
except ImportError:
    raise RuntimeError("Install the frozen package in the selected notebook environment.") from None
if wise.__version__ != "1.0.1":
    raise RuntimeError("Select the kernel with the frozen 1.0.1 package.")
try:
    from wise.evaluation import paper, native, replay, DataNotAvailable
except ImportError:
    raise RuntimeError("Complete the evaluation package installation and restart the kernel.") from None
if not os.environ.get("WISE_PAPER_DATA"):
    raise RuntimeError("Set WISE_PAPER_DATA before starting the notebook.")
NOTEBOOK_ROOT = Path.cwd().resolve()
if NOTEBOOK_ROOT.name != "notebooks":
    raise RuntimeError("Start this notebook in its notebooks folder.")
SAVE_OUTPUTS = True


In [ ]:
APPLICATION = 'bpic2013_incidents'
CONTEXT_COLUMNS = ['product_group']
GROUPINGS = ['assessment_group', 'product_group']
GAMMAS = (0.0, 5.0, 20.0, 100.0)
SHORTLIST_CAPACITIES = (5, 10, 20)
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs" / APPLICATION

def save_table(table, stem):
    if SAVE_OUTPUTS:
        target = OUTPUT_DIR.resolve()
        if not target.is_relative_to(NOTEBOOK_ROOT):
            raise RuntimeError("Output folder must stay inside notebooks.")
        target.mkdir(parents=True, exist_ok=True)
        table.to_csv(target / (stem + ".csv"))

def show_figure(figure, stem):
    if SAVE_OUTPUTS:
        target = OUTPUT_DIR.resolve()
        if not target.is_relative_to(NOTEBOOK_ROOT):
            raise RuntimeError("Output folder must stay inside notebooks.")
        target.mkdir(parents=True, exist_ok=True)
        figure.savefig(target / (stem + ".png"), dpi=180, bbox_inches="tight")
        figure.savefig(target / (stem + ".svg"), bbox_inches="tight")
    display(figure)
    plt.close(figure)

try:
    app = paper.application(APPLICATION)
except DataNotAvailable:
    raise RuntimeError("The verified public input is unavailable; no substitute result is generated.") from None
assessment = app.assessment
adapter_configuration_sha256 = hashlib.sha256(json.dumps(dict(assessment.configuration), sort_keys=True, separators=(",", ":")).encode()).hexdigest()
display({"adapter_configuration_sha256": adapter_configuration_sha256})
display(dict(assessment.source))
display(paper.incident_criteria())

# JSON presentation preserves values and represents compound keys as labels.
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


## 1. Baseline checks and numerical consistency

Compare this dataset's printed results through the library's comparison function. The reference arithmetic check tests the scoring and prioritisation of **prepared assessments** under both modes and all views. It does not validate whether the chosen norms are appropriate business policies, whether the raw operators match every generic constraint family, or whether an intervention improves outcomes.


In [ ]:
baseline_comparison = paper.compare(items=[26, 29, 30, 37, 38])
display(baseline_comparison)
baseline_summary = paper.summary(baseline_comparison)
display(baseline_summary)
for differences in baseline_summary["differ"]:
    if differences:
        raise AssertionError("A dataset baseline differs from the printed result.")
save_table(baseline_comparison, "paper_comparison")
verification = replay.consistency(assessment, app.result)
display(verification)
assert verification["passed"], "Prepared-assessment consistency check failed."


## 2. Preserve whole-case context

Attach the adapter's existing case attributes and rescore through the library. All recorded evidence of each case stays together. Regrouping changes the investigation unit, not the events or the underlying case assessments. The comparison reference remains the full scored population of the selected view.

No event-month filter is applied. Case start month, case end month and duration band require an explicit library context API; they must never be silently interchanged.


In [ ]:
context = assessment.features[CONTEXT_COLUMNS]
results = {
    mode: native.replay(assessment, attributes=context, mode=mode)
    for mode in ("flat", "layer_balanced")
}
primary = results["flat"]
display(primary)
context_replay_residual = replay.residual(primary.scores, app.result.scores)
display({"whole_case_context_replay_residual": context_replay_residual})
assert context_replay_residual < replay.TOLERANCE
for mode, result in results.items():
    display({"mode": mode, "decomposition_residual": result.check_decomposition()})
display(primary.frame().head(20))


## 3. Full-population investigation lists under each view

The table distinguishes scored volume, mean score, gap from the population reference and priority. These are relative investigation priorities, not breach counts, expected savings or causal effects. The bar chart renders the library's first ten rows; zero priorities may appear when fewer than ten slices have a positive priority.


In [ ]:
backlogs = {}
for view in primary.views:
    backlog = wise.prioritize(primary, by=native.GROUP, view=view)
    backlogs[("flat", view, 0.0)] = backlog
    display(Markdown("### View: " + view))
    display(backlog.spec.to_dict())
    display(backlog.table.head(10))
    save_table(backlog.table, "primary_" + view)

fig, axes = plt.subplots(1, len(primary.views), figsize=(15, 5), squeeze=False)
for ax, view in zip(axes[0], primary.views):
    table = backlogs[("flat", view, 0.0)].table.head(10)
    ax.barh([str(key) for key in table.index], table["stable_PI"])
    ax.invert_yaxis()
    ax.set_title(view)
    ax.set_xlabel("Priority Index")
fig.suptitle("Extended evaluation: investigation lists by view")
fig.tight_layout()
show_figure(fig, "extended_view_priorities")


## 4. View sensitivity and shortlist capacity

Use the library's Jaccard similarity of the top positive-priority slices. Capacity is an upper limit: a list may contain fewer positive slices. A value of 1 means identical eligible sets, not validated business importance; the same set can appear in a different order. The measure does not establish confidence intervals or rank stability under resampling.


In [ ]:
view_similarity = {}
for capacity in SHORTLIST_CAPACITIES:
    view_similarity[capacity] = {}
    for left, right in combinations(primary.views, 2):
        view_similarity[capacity][left + " / " + right] = wise.top_k_overlap(
            backlogs[("flat", left, 0.0)], backlogs[("flat", right, 0.0)], k=capacity
        )
positive_shortlists = {
    view: {str(capacity): backlog.table.loc[backlog.table["stable_PI"] > 0].head(capacity).index.to_list()
           for capacity in SHORTLIST_CAPACITIES}
    for view, backlog in ((view, backlogs[("flat", view, 0.0)]) for view in primary.views)
}
display({"positive_shortlist_keys": positive_shortlists})
display(view_similarity)

capacity = 10
view_matrix = [
    [wise.top_k_overlap(backlogs[("flat", left, 0.0)], backlogs[("flat", right, 0.0)], k=capacity)
     for right in primary.views]
    for left in primary.views
]
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(view_matrix, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(primary.views)), primary.views, rotation=25, ha="right")
ax.set_yticks(range(len(primary.views)), primary.views)
ax.set_title("Extended evaluation: positive top-10 Jaccard")
fig.colorbar(image, ax=ax, label="Shortlist similarity (Jaccard)")
fig.tight_layout()
show_figure(fig, "extended_view_similarity")


## 5. Small-slice shrinkage sensitivity

Compare γ = 0, 5, 20 and 100. These are declared scenarios, not estimated optimal settings. Increasing γ pulls small-slice means toward the selected full-population reference. Show the settings beside every list and use the library's overlap measure rather than recomputing a ranking in the notebook.


In [ ]:
gamma_similarity = {}
for view in primary.views:
    gamma_similarity[view] = {}
    for gamma in GAMMAS:
        backlog = wise.prioritize(primary, by=native.GROUP, view=view, gamma=gamma)
        backlogs[("flat", view, gamma)] = backlog
        gamma_similarity[view][str(gamma)] = {
            str(capacity): wise.top_k_overlap(backlogs[("flat", view, 0.0)], backlog, k=capacity)
            for capacity in SHORTLIST_CAPACITIES
        }
        display({"view": view, "gamma": gamma, "specification": backlog.spec.to_dict()})
        display(backlog.table.head(10))
        save_table(backlog.table, "gamma_" + view + "_" + str(gamma))
display(gamma_similarity)


### Investigation order across the declared gamma scenarios

Plot the first two baseline investigations under each view using the library's stabilised priorities at the four tested gamma values. Lines connect those scenario outputs; they do not estimate untested outcomes. This makes a change in the first investigation visible even when shortlist membership is identical.

In [ ]:
gamma_colors = {}
for view in primary.views:
    for key in backlogs[("flat", view, 0.0)].table.head(2).index:
        gamma_colors.setdefault(str(key), "C" + str(len(gamma_colors) % 10))
fig, axes = plt.subplots(len(primary.views), 1, figsize=(10, 10), squeeze=False)
for ax, view in zip(axes[:, 0], primary.views):
    keys = backlogs[("flat", view, 0.0)].table.head(2).index
    for key in keys:
        priorities = [backlogs[("flat", view, gamma)].table.loc[key, "stable_PI"] for gamma in GAMMAS]
        ax.plot(GAMMAS, priorities, marker="o", label=str(key), color=gamma_colors[str(key)])
    ax.set_title(view)
    ax.set_xticks(GAMMAS)
    ax.set_ylabel("Stabilised priority")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
axes[-1, 0].set_xlabel("Declared gamma scenario")
fig.suptitle("Extended evaluation: investigation order and shrinkage")
fig.tight_layout()
show_figure(fig, "extended_gamma_priorities")

# Membership stability is distinct from the investigation order above.
capacity = 10
membership_values = [[gamma_similarity[view][str(gamma)][str(capacity)] for gamma in GAMMAS] for view in primary.views]
fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(membership_values, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(GAMMAS)), [str(gamma) for gamma in GAMMAS])
ax.set_yticks(range(len(primary.views)), primary.views)
for row in range(len(primary.views)):
    for column in range(len(GAMMAS)):
        ax.text(column, row, format(membership_values[row][column], ".3f"), ha="center", va="center", color="white" if membership_values[row][column] > 0.6 else "black")
ax.set_xlabel("Declared gamma scenario")
ax.set_title("Extended evaluation: positive top-10 membership vs gamma 0")
fig.colorbar(image, ax=ax, label="Shortlist similarity (Jaccard)")
fig.tight_layout()
show_figure(fig, "extended_gamma_similarity")


## 6. Evidence-renormalisation sensitivity

Run flat and layer-balanced scoring on the same prepared assessments. The library's layer-balanced mode preserves each layer's total raw criterion weight; it does not assign equal weight to all layers. The modes coincide when every criterion is evaluated. Differences probe renormalisation over the evaluated set, jointly reflecting business scope and unavailable evidence. Inspect both when interpreting a change; this is a scoring-mode scenario with the original criterion definitions.


In [ ]:
mode_similarity = {}
for view in primary.views:
    flat = backlogs[("flat", view, 0.0)]
    balanced = wise.prioritize(results["layer_balanced"], by=native.GROUP, view=view)
    backlogs[("layer_balanced", view, 0.0)] = balanced
    mode_similarity[view] = {
        str(capacity): wise.top_k_overlap(flat, balanced, k=capacity)
        for capacity in SHORTLIST_CAPACITIES
    }
    display({"mode": "layer_balanced", "view": view, "specification": balanced.spec.to_dict()})
    display(balanced.table.head(10))
    save_table(balanced.table, "layer_balanced_" + view)
display(mode_similarity)


## 7. Alternative whole-case groupings

Different groupings answer different investigation questions and can merge heterogeneous cases. Keep the primary grouping visible and inspect each returned specification. Do not compare top-k sets across incompatible keys. Scores and the full-population reference are unchanged by regrouping.


In [ ]:
grouping_backlogs = {}
for grouping in GROUPINGS:
    label = grouping if isinstance(grouping, str) else " + ".join(grouping)
    grouping_backlogs[label] = {}
    for view in primary.views:
        backlog = wise.prioritize(primary, by=grouping, view=view)
        grouping_backlogs[label][view] = backlog
        display({"grouping": label, "view": view, "specification": backlog.spec.to_dict()})
        display(backlog.table.head(10))
        save_table(backlog.table, "grouping_" + label.replace(" + ", "_") + "_" + view)


## 8. Coverage as a separate evidence-investigation question

Coverage measures the share of in-scope raw weight supported by usable evidence. It is not log correctness or process quality. Aggregate the **library-returned coverage** with the library's generic prioritisation API. The renamed mean column is mean evidence coverage, and the renamed index weights the below-population coverage gap by case count.

This is an evidence-investigation list. It must not be interpreted as the paper's process-violation priority. Its population contains cases with defined coverage, including cases that may lack a process score. Undefined coverage remains excluded by the library.


In [ ]:
coverage_backlogs = {}
coverage_columns = {
    "mean_score": "mean_evidence_coverage", "global_mean": "population_mean_evidence_coverage",
    "gap": "coverage_gap", "PI": "evidence_investigation_index",
    "stable_mean": "stable_mean_evidence_coverage", "stable_gap": "stable_coverage_gap",
    "stable_PI": "stable_evidence_investigation_index",
}
for view in primary.views:
    coverage = primary.coverage(view)
    display({"view": view, "case_coverage": coverage.head(20).to_dict()})
    coverage_frame = coverage.to_frame("score").join(primary.cases[[native.GROUP]])
    backlog = wise.prioritize(coverage_frame, by=native.GROUP)
    coverage_backlogs[view] = backlog
    display({"view": view, "evidence_investigation_settings": backlog.spec.to_dict()})
    display(backlog.table.rename(columns=coverage_columns).head(10))
    save_table(backlog.table.rename(columns=coverage_columns), "evidence_coverage_" + view)


### Criterion-level evidence and severity summaries

Aggregate the library's evaluated flags within business scope. A displayed availability mean is the fraction of in-scope cases whose criterion can be evaluated, while the library count is the number of in-scope cases. This is an unweighted per-criterion diagnostic, distinct from weighted case coverage.

Separately aggregate the returned severity values over evaluated cases only. Display its mean and count; do not interpret the generic prioritisation columns for this diagnostic. Selection, boolean-to-number conversion and column labels do not recalculate the criterion severities or implement an aggregation formula.

In [ ]:
criterion_availability = {}
criterion_severity = {}
for criterion in assessment.criteria:
    availability = primary.evaluated[criterion].where(primary.in_scope[criterion]).astype(float)
    availability_frame = availability.to_frame("score").join(primary.cases[[native.GROUP]])
    severity_frame = primary.violations[criterion].to_frame("score").join(primary.cases[[native.GROUP]])
    try:
        availability_backlog = wise.prioritize(availability_frame, by=native.GROUP)
        severity_backlog = wise.prioritize(severity_frame, by=native.GROUP)
    except wise.NotScoredError:
        display({"criterion": criterion, "status": "no eligible observations for this diagnostic"})
        continue
    availability_table = availability_backlog.table[["n_cases", "mean_score", "global_mean"]].rename(columns={
        "n_cases": "in_scope_cases", "mean_score": "evaluable_fraction", "global_mean": "population_evaluable_fraction"})
    severity_table = severity_backlog.table[["n_cases", "mean_score", "global_mean"]].rename(columns={
        "n_cases": "evaluated_cases", "mean_score": "mean_severity", "global_mean": "population_mean_severity"})
    criterion_availability[criterion] = availability_table
    criterion_severity[criterion] = severity_table
    display({"criterion": criterion})
    display(availability_table.sort_index())
    display(severity_table.sort_index())
    save_table(availability_table, "criterion_availability_" + criterion)
    save_table(severity_table, "criterion_severity_" + criterion)


## 9. Explain signed layer drivers

Display the signed layer contributions and mean layer drivers for the primary top slices. Positive terms increase the relative penalty gap; negative terms offset it. Keep both signs visible. The library's consistency test checks the decomposition, including clipping after summation. Layer explanations direct evidence inspection; they do not identify causes.


In [ ]:
for view in primary.views:
    backlog = backlogs[("flat", view, 0.0)]
    signed = wise.layer_contributions(primary, backlog)
    drivers = wise.layer_drivers(primary, backlog)
    keys = backlog.table.head(10).index
    display(Markdown("### Signed layer drivers: " + view))
    display(signed.loc[keys])
    display(drivers.loc[keys])
    save_table(signed, "signed_layers_" + view)
    save_table(drivers, "layer_drivers_" + view)

view = primary.views[0]
backlog = backlogs[("flat", view, 0.0)]
keys = backlog.table.head(2).index
signed = wise.layer_contributions(primary, backlog)
fig, axes = plt.subplots(len(keys), 1, figsize=(9, 7), sharex=True, squeeze=False)
for ax, key in zip(axes[:, 0], keys):
    values = signed.loc[key]
    ax.barh(list(values.index), values.to_list())
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(str(key) + " / " + view)
axes[-1, 0].set_xlabel("Signed contribution to assessed priority")
fig.suptitle("Extended evaluation: signed drivers and offsets")
fig.tight_layout()
show_figure(fig, "extended_signed_layer_drivers")


## 10. Case evidence drill-down

Select an individual case from the library's case-level priority list, then inspect its complete prepared feature record, raw criterion values, evidence states, severities, effective weights and penalties. This selection uses relative priority; a large case penalty may warrant attention even when its case-level relative priority is zero.

The attached native log is an **assessment carrier**, with one placeholder record per case. Its trace is not the original process execution. These evidence tables must not be presented as a reconstructed event trace. Original record witnesses require a library evidence-retrieval API.


In [ ]:
case_backlog = wise.prioritize(primary, by="case_id", view=primary.views[0])
display(case_backlog.table.head(10))
selected_cases = [case_backlog.table.index[0]]
paper_witness = '1-530458987'
if paper_witness is not None and paper_witness in assessment.features.index:
    selected_cases.append(paper_witness)
case_evidence = {}
for case_id in selected_cases:
    display({"case_id": case_id, "prepared_features": assessment.features.loc[case_id].to_dict()})
    display({"raw_criterion_values": assessment.raw.loc[case_id].to_dict()})
    display({"evidence_states": assessment.state.loc[case_id].to_dict()})
    display({"scope": assessment.in_scope.loc[case_id].to_dict(),
             "severities": assessment.severity.loc[case_id].to_dict()})
    case_evidence[str(case_id)] = {
        "features": assessment.features.loc[case_id].to_dict(),
        "raw": assessment.raw.loc[case_id].to_dict(),
        "states": assessment.state.loc[case_id].to_dict(),
        "scope": assessment.in_scope.loc[case_id].to_dict(),
        "severity": assessment.severity.loc[case_id].to_dict(),
    }
    for view in primary.views:
        display({"view": view, "score": primary.scores.loc[case_id, view],
                 "coverage": primary.coverage(view).loc[case_id],
                 "effective_weights": primary.effective_weights(view).loc[case_id].to_dict(),
                 "penalties": primary.penalties(view).loc[case_id].to_dict()})


## 11. Additional API integrity tests

Check that invalid shrinkage and unknown context keys are refused, explanations cannot silently switch views, and comparisons cannot mix incompatible grouping keys. These are contract tests of the installed library, not fabricated event-log experiments.


In [ ]:
contract_checks = {}
def expect_norm_error(label, operation):
    try:
        operation()
    except wise.NormError:
        contract_checks[label] = "passed"
    else:
        raise AssertionError("Expected a rejected invalid configuration: " + label)

expect_norm_error("negative gamma", lambda: wise.prioritize(primary, by=native.GROUP, view=primary.views[0], gamma=-1))
expect_norm_error("non-finite gamma", lambda: wise.prioritize(primary, by=native.GROUP, view=primary.views[0], gamma=float("nan")))
expect_norm_error("unknown view", lambda: wise.prioritize(primary, by=native.GROUP, view="undefined_view"))
expect_norm_error("unknown whole-case context", lambda: wise.prioritize(primary, by="undefined_context", view=primary.views[0]))
expect_norm_error("explanation view mismatch", lambda: wise.layer_contributions(primary, backlogs[("flat", primary.views[0], 0.0)], view=primary.views[1]))
other_grouping = GROUPINGS[1]
other_backlog = wise.prioritize(primary, by=other_grouping, view=primary.views[0])
expect_norm_error("incompatible overlap grouping", lambda: wise.top_k_overlap(backlogs[("flat", primary.views[0], 0.0)], other_backlog))
display(contract_checks)


### Actual-data metamorphic tests

Two additional tests use the verified assessments: reverse the order of complete case records and reload the serialised norm. Both should preserve case identity and scores. These test order independence and portable norm semantics without creating substitute empirical data.

In [ ]:
permuted_assessment = replace(assessment,
    features=assessment.features.iloc[::-1], severity=assessment.severity.iloc[::-1],
    in_scope=assessment.in_scope.iloc[::-1], state=assessment.state.iloc[::-1],
    raw=assessment.raw.iloc[::-1], groups=assessment.groups.iloc[::-1])
permuted_result = native.replay(permuted_assessment, attributes=context.iloc[::-1])
permutation_residual = replay.residual(permuted_result.scores, primary.scores)
assert permutation_residual < replay.TOLERANCE
roundtrip_norm = wise.Norm.loads(primary.norm.dumps())
roundtrip_result = wise.score(primary.log, roundtrip_norm)
roundtrip_residual = replay.residual(roundtrip_result.scores, primary.scores)
assert roundtrip_residual < replay.TOLERANCE
contract_checks["whole-case order independence"] = "passed"
contract_checks["serialised norm score roundtrip"] = "passed"
display({"whole_case_permutation_residual": permutation_residual,
         "norm_roundtrip_residual": roundtrip_residual})
del permuted_result, roundtrip_result, permuted_assessment


## 12. Process improvement interpretation and next evidence

Use routing and timing views to separate reassignment investigations from long recorded spans. Confirm whether repeated states or a return to a prior team represent avoidable work before proposing an intervention.

Separate four claims when presenting these experiments: numerical consistency, descriptive investigation value, robustness to settings, and measured business improvement. Only the first three are addressed here. The last needs verified policies, outcome labels, domain review and an intervention design.

CSV tables and extension figures are written to the ignored `outputs` folder. Clear notebook outputs before sharing or committing; never store raw source logs, the unpublished paper text or a data-directory value in the notebook.


In [ ]:
experiment_record = {
    "application": APPLICATION,
    "package_version": wise.__version__,
    "norm_fingerprint": primary.norm_fingerprint,
    "adapter_configuration_sha256": adapter_configuration_sha256,
    "positive_shortlists": positive_shortlists,
    "source": dict(assessment.source),
    "primary_grouping": native.GROUP,
    "views": primary.views,
    "gammas": GAMMAS,
    "shortlist_capacities": SHORTLIST_CAPACITIES,
    "view_similarity": view_similarity,
    "gamma_similarity": gamma_similarity,
    "mode_similarity": mode_similarity,
    "whole_case_context_replay_residual": context_replay_residual,
    "verification": verification,
    "contract_checks": contract_checks,
    "primary_top_rows": {view: backlogs[("flat", view, 0.0)].table.head(10).to_dict("index") for view in primary.views},
    "gamma_top_rows": {view: {str(gamma): backlogs[("flat", view, gamma)].table.head(10).to_dict("index") for gamma in GAMMAS} for view in primary.views},
    "mode_top_rows": {view: backlogs[("layer_balanced", view, 0.0)].table.head(10).to_dict("index") for view in primary.views},
    "coverage_top_rows": {view: coverage_backlogs[view].table.head(10).to_dict("index") for view in primary.views},
    "grouping_top_rows": {key: {view: backlog.table.head(10).to_dict("index") for view, backlog in values.items()} for key, values in grouping_backlogs.items()},
    "case_evidence": case_evidence,
    "criterion_availability": {key: table.to_dict("index") for key, table in criterion_availability.items()},
    "criterion_severity": {key: table.to_dict("index") for key, table in criterion_severity.items()},
    "metamorphic_residuals": {"whole_case_permutation": permutation_residual, "norm_roundtrip": roundtrip_residual},
}
if SAVE_OUTPUTS:
    target = OUTPUT_DIR.resolve()
    if not target.is_relative_to(NOTEBOOK_ROOT):
        raise RuntimeError("Output folder must stay inside notebooks.")
    target.mkdir(parents=True, exist_ok=True)
    (target / "experiment_record.json").write_text(json.dumps(json_ready(experiment_record), indent=2, default=str, allow_nan=False), encoding="utf-8")
display({"evaluation_complete": True, "prepared_assessment_checks_passed": verification["passed"], "contract_checks": contract_checks})


## Additional source-log investigation: problem management

The previous sections assess the prepared incident baseline. This supplement reads the **two raw problem logs** and preserves that distinction: problem-management goals concern investigation, useful workarounds and durable causes, rather than restoring every incident quickly or forcing every case to the first support line. Do not compare their priority values directly with the prepared incident assessment, which has different criteria and evidence preparation.

The [owner’s challenge description](https://ais.win.tue.nl/bpi/2013/challenge.html) provides incident, open-problem and closed-problem extracts from VINST. It limits the push-to-front question to incidents. The log filename describes a selected population, not a verified lifecycle for every trace. `Completed` observations need their lifecycle and source context before being interpreted as closure.

In [ ]:
import os
import matplotlib.pyplot as plt
from dataclasses import replace
import pandas as pd
import wise
from IPython.display import display, Markdown
from bpic_analysis import (
    challenge, summary, norm_table, criterion_summary, activity_summary,
    context_profiles, priorities, plot_overview, case_evidence,
    threshold_sensitivity, view_agreement, integrity, journal, explanations,
)

source_runs_2013 = challenge(2013, data_folder=os.environ.get("WISE_PAPER_DATA"))
problem_runs = {name: run for name, run in source_runs_2013.items()
                if name in ("bpic2013_open_problems", "bpic2013_closed_problems")}
display(summary(problem_runs))
display(journal(problem_runs))

### PM² extraction and data processing: keep the source semantics

Retain all observations of each selected problem. Queue and acceptance labels are evidence of recorded actions, not proven handoffs or total queue residence. Repeated `Completed` observations do not establish repeated failure. The first acceptance-to-completion lag does not measure final lifecycle closure. A missing country field or a changing product label must remain visible in the context review.

In [ ]:
for name, run in problem_runs.items():
    if run.get("result") is None:
        continue
    display(Markdown(f"### {name}"))
    display(activity_summary(run, labels=["Queued", "Accepted", "Completed"]))
    display(context_profiles(run, attributes=["product", "impact", "organization country", "oranization country", "case_start_month", "observed_span_band"]))
    display(norm_table(run))
    display(run["specification"].get("pending_checks", []))

### PM² mining and analysis: compare investigation leads with their coverage

Review initial queue-to-acceptance timing, acceptance-to-first-completion timing and observation intensity. These diagnostic signals do not measure root-cause quality or whether a workaround prevents further incidents. Compare each selected population with its own reference. Do not add their totals or interpret a score difference as proof that open problems are managed worse.

The library renormalises the score over usable evidence. A high score cannot certify adherence to the full norm when some evidence is unavailable. Criterion coverage is the unweighted fraction of in-scope cases that can be evaluated for that criterion. The case witness also shows each view’s weighted evidence coverage. These are different measures, and neither certifies that the source contains every real process event.

In [ ]:
problem_backlogs = {}
for name, run in problem_runs.items():
    if run.get("result") is None:
        continue
    display(Markdown(f"### {name}"))
    display(criterion_summary(run))
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=20)
        problem_backlogs[(name, view)] = backlog
        display(Markdown(f"#### {view}"))
        display(backlog.table.head(10))
        explanation = explanations(run, backlog)
        display(explanation["contributions"].reindex(backlog.table.head(10).index))
        display(explanation["drivers"].reindex(backlog.table.head(10).index))
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)

### Preference, threshold and aggregation sensitivity

Ask whether the same problem remains worth investigating when the allowed diagnostic intervals change or the view favours timing over coordination. Keep a long-running, actively investigated problem distinct from one repeatedly transferred without a useful accepting owner. Rescore complete traces; do not break them into event-month fragments.

Sensitivity changes lag threshold and width together and occurrence severity width `K` with allowed count `k` fixed. The derived case-start month uses UTC and is an observed timestamp cohort, not a verified business-local policy period.

In [ ]:
for name, run in problem_runs.items():
    if run.get("result") is None:
        continue
    display(Markdown(f"### {name}"))
    display(threshold_sensitivity(run, factors=(0.5, 1, 2)))
    display(view_agreement(run, k=10))
    for context_key in ("product", "impact", "case_start_month"):
        if context_key in run["result"].cases.columns and not run["log"].varying_case_attributes.get(context_key, 0):
            display(priorities(run, by=context_key, gamma=20).table.head(10))
    view = run["result"].views[0]
    reference = priorities(run, view=view)
    for mode in ("flat", "layer_balanced"):
        source_result = replace(wise.score(run["log"], run["norm"], mode=mode), cases=run["result"].cases)
        alternative = wise.prioritize(source_result, by=list(reference.spec.by), view=view)
        display({"dataset": name, "mode": mode,
                 "positive_top10_overlap": wise.top_k_overlap(reference, alternative, k=10)})

### PM² evaluation: inspect an investigation case and its unknowns

Inspect the complete problem history, context and criterion evaluation. Check whether repeat queue or completion observations represent specialist work, administrative updates or a return caused by missing information. Ask where the workaround, investigation owner and expected next action are recorded. These logs alone may not show whether a durable fix reduced later incidents.

In [ ]:
problem_witnesses = {}
for name, run in problem_runs.items():
    if run.get("result") is None:
        continue
    problem_witnesses[name] = case_evidence(run)
    display(Markdown(f"### {name}"))
    for label, value in problem_witnesses[name].items():
        display(Markdown(f"**{label}**"))
        display(value)
    display(integrity(run))

### PM² process improvement and support: improve the investigation mechanism

A problem-management lead calls for an investigation owner and a useful next action, not indiscriminate faster closure. Use a pilot only after reviewing the full case. Preserve necessary specialist involvement and monitor whether the proposed mechanism improves the service outcome.

In [ ]:
problem_improvements = pd.DataFrame([
    {"Issue to confirm": "Problem transfers lack a clear accepting owner", "Pilot": "Assign one investigation owner and record acceptance and next action at each specialist handoff", "Owner": "Problem-management lead", "Business outcome": "Fewer preventable returns and clearer progress to a usable workaround or durable fix", "Safeguard": "Preserve required specialist work and do not close unresolved causes to reduce a score"},
    {"Issue to confirm": "Long investigations have no useful review point", "Pilot": "Agree a workaround review date and a specific evidence request for one recurring problem family", "Owner": "Service owner", "Business outcome": "Better workaround availability and fewer linked repeat incidents where valid links exist", "Safeguard": "Measure related incident outcomes with verified relationships; time alone does not establish investigation quality"},
])
display(problem_improvements)
problem_decision_journal = pd.DataFrame([
    {"PM² stage": "Planning", "Question": "Confirm root-cause and workaround goals for the two selected populations", "Status": "Pending owner review"},
    {"PM² stage": "Extraction and data processing", "Question": "Verify population selection, lifecycle closure and changing context", "Status": "Review source journal and witnesses"},
    {"PM² stage": "Mining and analysis", "Question": "Choose a supported lead with adequate coverage and explain its preference sensitivity", "Status": "Review computed tables"},
    {"PM² stage": "Evaluation", "Question": "Distinguish necessary investigation, avoidable transfer and recording gaps", "Status": "Pending case review"},
    {"PM² stage": "Process improvement and support", "Question": "Agree owner, pilot outcome, quality safeguard and follow-up", "Status": "No pilot undertaken"},
])
display(journal(problem_runs))
display(problem_decision_journal)